In [2]:

import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import re
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import densenet121
from torch.nn.utils.rnn import pad_sequence
import math


In [3]:

# ----------------------------
# ✅ Force CPU
# ----------------------------
device = torch.device("cpu")


In [4]:

# ----------------------------
# ✅ Paths
# ----------------------------
TRAIN_PATH = "_Images/_Images_normalized/div_images/train"
VALID_PATH = "_Images/_Images_normalized/div_images/valid"
TEST_PATH  = "_Images/_Images_normalized/div_images/test"




In [5]:
# ----------------------------
# ✅ Load and Filter CSV (remove duplicates + check existence)
# ----------------------------
def load_and_filter_csv(csv_path, image_dir):
    df = pd.read_csv(csv_path)
    df['filename'] = df['filename'].astype(str).str.replace('\\', '/', regex=False).str.strip()
    df = df.drop_duplicates(subset='filename')  # remove duplicates
    df['full_path'] = df['filename'].apply(lambda f: os.path.join(image_dir, f))
    df = df[df['full_path'].apply(os.path.exists)].copy()
    return df.drop(columns=['full_path'])

train_df = load_and_filter_csv("_Images/_Images_normalized/train_split.csv", TRAIN_PATH)
valid_df = load_and_filter_csv("_Images/_Images_normalized/valid_split.csv", VALID_PATH)
test_df  = load_and_filter_csv("_Images/_Images_normalized/test_split.csv", TEST_PATH)


In [6]:

# ----------------------------
# ✅ Tokenizer and Vocabulary
# ----------------------------
def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\d+", "", text)
    return text.strip().split()

def build_vocab(df):
    df['report'] = df['findings'].fillna('') + ' ' + df['impression'].fillna('') 
    tokenized_reports = [tokenize(r) for r in df['report']]
    special_tokens = ["<pad>", "<start>", "<end>", "<unk>"]
    all_tokens = [w for tokens in tokenized_reports for w in tokens]
    vocab = special_tokens + [w for w, c in Counter(all_tokens).items() if c >= 2]
    word2idx = {word: idx for idx, word in enumerate(vocab)}
    idx2word = {idx: word for word, idx in word2idx.items()}
    return word2idx, idx2word

word2idx, idx2word = build_vocab(train_df)


In [7]:

# ----------------------------
# ✅ Transformations
# ----------------------------
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])


In [8]:

# ----------------------------
# ✅ Dataset for Images Only
# ----------------------------
class CXRImageOnlyDataset(Dataset):
    def __init__(self, image_dir, dataframe, transform=None):
        self.image_dir = image_dir
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.image_dir, row['filename'])
        image = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise ValueError(f"Could not read image: {path}")
        image = np.stack([image] * 3, axis=-1)
        if self.transform:
            image = self.transform(image)
        return image

In [9]:

class CXRReportDataset(Dataset):
    def __init__(self, image_dir, dataframe, word2idx, transform=None):
        self.image_dir = image_dir
        self.df = dataframe
        self.word2idx = word2idx
        self.transform = transform
        self.df['report'] = self.df['findings'].fillna('') + ' ' + self.df['impression'].fillna('')
        self.df['filename'] = self.df['filename'].astype(str).str.replace('\\', '/', regex=False).str.strip()

    def encode_report(self, text):
        tokens = tokenize(text)
        return [self.word2idx["<start>"]] + [self.word2idx.get(t, self.word2idx["<unk>"]) for t in tokens] + [self.word2idx["<end>"]]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.image_dir, row['filename'])
        image = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise ValueError(f"Could not read image: {path}")
        image = np.stack([image] * 3, axis=-1)
        if self.transform:
            image = self.transform(image)
        report_tensor = torch.tensor(self.encode_report(row['report']), dtype=torch.long)
        return image, report_tensor


In [10]:
# ----------------------------
# ✅ Dataset for Reports
# ----------------------------
train_dataset_reports = CXRReportDataset(TRAIN_PATH, train_df, word2idx, transform=transform)
valid_dataset_reports = CXRReportDataset(VALID_PATH, valid_df, word2idx, transform=transform)
test_dataset_reports  = CXRReportDataset(TEST_PATH, test_df, word2idx, transform=transform)


In [11]:
# ----------------------------
# ✅ Pad Collate Function
# ----------------------------
def collate_fn(batch):
    images, reports = zip(*batch)
    images = torch.stack(images)
    reports = [torch.tensor(r, dtype=torch.long) for r in reports]
    padded_reports = pad_sequence(reports, batch_first=False, padding_value=word2idx["<pad>"])
    return images, padded_reports


In [12]:

# ----------------------------
# ✅ Datasets & Loaders
# ----------------------------
train_dataset_img_only = CXRImageOnlyDataset(TRAIN_PATH, train_df, transform=transform)
valid_dataset_img_only = CXRImageOnlyDataset(VALID_PATH, valid_df, transform=transform)
test_dataset_img_only  = CXRImageOnlyDataset(TEST_PATH, test_df, transform=transform)

train_loader_img = DataLoader(train_dataset_img_only, batch_size=32, shuffle=False)
valid_loader_img = DataLoader(valid_dataset_img_only, batch_size=32, shuffle=False)
test_loader_img  = DataLoader(test_dataset_img_only, batch_size=32, shuffle=False)


In [13]:
print("Train images:", len(train_dataset_img_only))
print("Valid images:", len(valid_dataset_img_only))
print("Test images:", len(test_dataset_img_only))


Train images: 4700
Valid images: 937
Test images: 651


In [14]:

# ----------------------------
# ✅ Feature Extractor
# ----------------------------
class DenseNetFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        densenet = densenet121(pretrained=True)
        self.features = densenet.features
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        return x.view(x.size(0), -1)  # [B, 1024]

feature_extractor = DenseNetFeatureExtractor().to(device).eval()


c:\Users\ag331\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\ag331\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [15]:

# ----------------------------
# ✅ Feature Extraction Function
# ----------------------------
def extract_features(loader, model, device):
    features = []
    count = 0
    with torch.no_grad():
        for batch_idx, images in enumerate(loader):
            try:
                images = images.to(device)
                feats = model(images)
                features.append(feats.cpu().numpy())
                count += images.size(0)
            except Exception as e:
                print(f"❌ Error in batch {batch_idx}: {e}")
    print(f"✅ Extracted features from {count} images.")
    return np.vstack(features)



In [16]:

# ----------------------------
# ✅ Run Extraction & Save
# ----------------------------
train_features = extract_features(train_loader_img, feature_extractor, device)
valid_features = extract_features(valid_loader_img, feature_extractor, device)
test_features  = extract_features(test_loader_img, feature_extractor, device)

np.save("train_features.npy", train_features)
np.save("valid_features.npy", valid_features)
np.save("test_features.npy", test_features)

print("✅ Feature extraction complete!")
print("Train:", train_features.shape)
print("Valid:", valid_features.shape)
print("Test :", test_features.shape)


✅ Extracted features from 4700 images.
✅ Extracted features from 937 images.
✅ Extracted features from 651 images.
✅ Feature extraction complete!
Train: (4700, 1024)
Valid: (937, 1024)
Test : (651, 1024)


In [17]:

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=49):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: [B, seq_len, d_model]
        x = x + self.pe[:, :x.size(1), :]
        return x
class TransformerImageEncoder(nn.Module):
    def __init__(self, input_dim=1024, d_model=512, nhead=8, num_layers=3, max_len=49):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)  # project 1024 → d_model (e.g. 512)
        self.pos_encoder = PositionalEncoding(d_model=d_model, max_len=max_len)

        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, x):
        # x: [B, 1024]
        x = x.unsqueeze(1)  # [B, 1, 1024]
        x = self.input_proj(x)  # [B, 1, d_model]
        x = self.pos_encoder(x)  # add positional encodings
        x = x.permute(1, 0, 2)  # → [seq_len=1, B, d_model] (required by Transformer)
        x = self.transformer_encoder(x)  # → [1, B, d_model]
        return x.permute(1, 0, 2)  # → [B, 1, d_model]

class LSTMRefiner(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=512, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim,
                            num_layers=num_layers, batch_first=True)

    def forward(self, x):
        # x: [B, seq_len=1, input_dim=512]
        output, (h_n, c_n) = self.lstm(x)  # output: [B, seq_len, hidden_dim]
        return output  # or return h_n if you just need the final state
   

In [18]:
import pandas as pd
import re
from collections import Counter
import os

# Load CSV (change path for train, valid, test as needed)
df = pd.read_csv("_Images/_Images_normalized/train_split.csv")  # Replace with valid.csv or test.csv accordingly

# Step 1: Combine 'findings' and 'impression'

df['report'] = df['findings'].fillna('') + ' ' + df['impression'].fillna('') 
# Step 2: Text Preprocessing & Tokenization (as per paper)
def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)  # remove punctuation
    text = re.sub(r"\d+", "", text)      # remove numbers
    return text.strip().split()

# Step 3: Tokenize all reports
tokenized_reports = [tokenize(r) for r in df['report']]

# Step 4: Build vocabulary (keep words that appear ≥ 2 times)
special_tokens = ["<pad>", "<start>", "<end>", "<unk>"]
all_tokens = [w for tokens in tokenized_reports for w in tokens]
vocab = special_tokens + [w for w, c in Counter(all_tokens).items() if c >= 2]

# Step 5: Create word <-> index mappings
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}

# Step 6: Encode a report
def encode_report(text):
    tokens = tokenize(text)
    return [word2idx["<start>"]] + [word2idx.get(t, word2idx["<unk>"]) for t in tokens] + [word2idx["<end>"]]

# Step 7: Decode back (for debugging)
def decode_report(indices):
    return " ".join(idx2word.get(i, "<unk>") for i in indices if i != word2idx["<pad>"])

# ✅ Optional: Test one encoded/decode report
example = df.iloc[0]['report']
encoded = encode_report(example)
print("Encoded:", encoded)
print("Decoded:", decode_report(encoded))



Encoded: [1, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 9, 16, 2]
Decoded: <start> borderline cardiomegaly midline sternotomy enlarged pulmonary arteries clear lungs inferior no acute pulmonary findings <end>


In [19]:


class TransformerReportDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=512, nhead=8, num_layers=3, max_len=100):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=max_len)

        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=nhead)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)

        self.fc_out = nn.Linear(d_model, vocab_size)
        self.d_model = d_model

    def forward(self, tgt, memory):
        # tgt: [T, B]         (token indices)
        # memory: [S, B, D]   (LSTM output, usually S=1)

        embedded = self.embedding(tgt) * (self.d_model ** 0.5)  # [T, B, D]
        embedded = self.pos_encoder(embedded)                  # [T, B, D]

        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(0)).to(tgt.device)

        output = self.transformer_decoder(embedded, memory, tgt_mask=tgt_mask)  # [T, B, D]
        output = self.fc_out(output)  # [T, B, vocab_size]
        return output

In [20]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
import numpy as np

# Assuming these are already defined:
# - train_dataset
# - TransformerImageEncoder, LSTMRefiner, TransformerReportDecoder
# - word2idx, encode_report

# ----------------------------
# ✅ Pad Collate Function
# ----------------------------
def collate_fn(batch):
    images, reports = zip(*batch)
    images = torch.stack(images)
    reports = [torch.tensor(r, dtype=torch.long) for r in reports]
    padded_reports = pad_sequence(reports, batch_first=False, padding_value=word2idx["<pad>"])
    return images, padded_reports

# ----------------------------
# ✅ Recreate DataLoader with tokenized reports
# ----------------------------
train_loader = DataLoader(train_dataset_reports, batch_size=16, shuffle=True, collate_fn=collate_fn)

# ----------------------------
# ✅ Initialize Model
# ----------------------------
encoder = TransformerImageEncoder().to(device)
lstm = LSTMRefiner().to(device)
decoder = TransformerReportDecoder(vocab_size=len(word2idx)).to(device)

# ----------------------------
# ✅ Optimizer & Loss
# ----------------------------
params = list(encoder.parameters()) + list(lstm.parameters()) + list(decoder.parameters())
optimizer = torch.optim.Adam(params, lr=1e-4)
criterion = nn.CrossEntropyLoss(ignore_index=word2idx["<pad>"])

# ----------------------------
# ✅ Training Loop
# ----------------------------
epochs = 30
for epoch in range(epochs):
    encoder.train()
    lstm.train()
    decoder.train()
    
    total_loss = 0
    for images, tgt in train_loader:
        images, tgt = images.to(device), tgt.to(device)  # tgt: [T, B]

        # Forward pass through encoder and LSTM
        features = encoder(feature_extractor(images))    # [B, 1, 512]
        memory = lstm(features)                           # [B, 1, 512]
        memory = memory.permute(1, 0, 2)                  # [S=1, B, D]

        # Decoder input is tgt[:-1]; target is tgt[1:]
        decoder_input = tgt[:-1, :]
        target = tgt[1:, :]

        output = decoder(decoder_input, memory)           # [T-1, B, vocab_size]
        loss = criterion(output.view(-1, output.size(-1)), target.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

print("\n✅ Training Complete!")

c:\Users\ag331\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
C:\Users\ag331\AppData\Local\Temp\ipykernel_43744\1565583725.py:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  reports = [torch.tensor(r, dtype=torch.long) for r in reports]


Epoch 1/30, Loss: 3.4812
Epoch 2/30, Loss: 2.4122
Epoch 3/30, Loss: 2.0799
Epoch 4/30, Loss: 1.8653
Epoch 5/30, Loss: 1.6861
Epoch 6/30, Loss: 1.5363
Epoch 7/30, Loss: 1.3995
Epoch 8/30, Loss: 1.2873
Epoch 9/30, Loss: 1.1725
Epoch 10/30, Loss: 1.0699
Epoch 11/30, Loss: 0.9749
Epoch 12/30, Loss: 0.8916
Epoch 13/30, Loss: 0.8053
Epoch 14/30, Loss: 0.7345
Epoch 15/30, Loss: 0.6715
Epoch 16/30, Loss: 0.6206
Epoch 17/30, Loss: 0.5715
Epoch 18/30, Loss: 0.5273
Epoch 19/30, Loss: 0.4848
Epoch 20/30, Loss: 0.4507
Epoch 21/30, Loss: 0.4262
Epoch 22/30, Loss: 0.3973
Epoch 23/30, Loss: 0.3775
Epoch 24/30, Loss: 0.3519
Epoch 25/30, Loss: 0.3303
Epoch 26/30, Loss: 0.3118
Epoch 27/30, Loss: 0.2962
Epoch 28/30, Loss: 0.2790
Epoch 29/30, Loss: 0.2611
Epoch 30/30, Loss: 0.2488

✅ Training Complete!


In [21]:
# Save the model state dicts
torch.save({
    'encoder_state_dict': encoder.state_dict(),
    'lstm_state_dict': lstm.state_dict(),
    'decoder_state_dict': decoder.state_dict(),
    'optimizer_state_dict': optimizer.state_dict()
}, 'report_generator_model.pth')

print("📦 Model saved as 'report_generator_model.pth'")


📦 Model saved as 'report_generator_model.pth'


In [27]:
def beam_search(image, encoder, lstm, decoder, word2idx, idx2word, feature_extractor, beam_width=3, max_len=100, temperature=1.0):
    encoder.eval()
    lstm.eval()
    decoder.eval()

    with torch.no_grad():
        image = image.unsqueeze(0).to(device)
        features = feature_extractor(image)
        enc_output = encoder(features)
        memory = lstm(enc_output)
        memory = memory.permute(1, 0, 2)

        sequences = [[word2idx["<start>"]]]
        scores = torch.zeros(1, device=device)

        alpha = 0.7  # Length normalization
        for _ in range(max_len):
            all_candidates = []
            for i, seq in enumerate(sequences):
                if seq[-1] == word2idx["<end>"]:
                    all_candidates.append((seq, scores[i]))
                    continue

                tgt_seq = torch.tensor(seq, dtype=torch.long, device=device).unsqueeze(1)
                out = decoder(tgt_seq, memory)
                log_probs = torch.log_softmax(out[-1, 0] / temperature, dim=-1)

                topk_log_probs, topk_idx = torch.topk(log_probs, beam_width)

                for j in range(beam_width):
                    new_seq = seq + [topk_idx[j].item()]
                    new_score = scores[i] + topk_log_probs[j]
                    all_candidates.append((new_seq, new_score))

            ordered = sorted(all_candidates, key=lambda tup: tup[1] / (len(tup[0]) ** alpha), reverse=True)
            sequences, scores = zip(*ordered[:beam_width])
            scores = torch.tensor(scores, device=device)

            if all(seq[-1] == word2idx["<end>"] for seq in sequences):
                break

        best_seq = sequences[0]
        return " ".join(idx2word[idx] for idx in best_seq if idx not in [word2idx["<start>"], word2idx["<end>"], word2idx["<pad>"]])

# ----------------------------
# ✅ Simple Inference Example (Validation)
# ----------------------------
print("\n📸 Testing some samples:\n")
for i in range(5):
    image, _ = valid_dataset_reports[i]
    generated_report = beam_search(image, encoder, lstm, decoder, word2idx, idx2word, feature_extractor, beam_width=3, max_len=50, temperature=0.8)
    print(f"Image #{i+1} Generated Report:\n{generated_report}\n")



📸 Testing some samples:

Image #1 Generated Report:
the lungs are clear bilaterally specifically no evidence of focal consolidation pneumothorax or pleural effusion cardio mediastinal silhouette is unremarkable visualized osseous structures of the thorax are without acute abnormality no acute cardiopulmonary abnormality

Image #2 Generated Report:
the lungs are clear bilaterally specifically no evidence of focal consolidation pneumothorax or pleural effusion cardio mediastinal silhouette is unremarkable visualized osseous structures of the thorax are without acute abnormality no acute cardiopulmonary abnormality

Image #3 Generated Report:
the heart is normal in size the mediastinum is unremarkable the lungs are clear no acute disease

Image #4 Generated Report:
the cardiomediastinal silhouette is within normal limits for size and contour the lungs are normally inflated without evidence of focal airspace disease pleural effusion or pneumothorax cholecystectomy clips elevation the righ

In [28]:
generated_reports = []
ground_truth_reports = []

for i in range(10):
    image, _ = valid_dataset_reports[i]
    print(f"\n📸 Image #{i+1}")

    print("Generated:", beam_search(
        image=image,
        encoder=encoder,
        lstm=lstm,
        decoder=decoder,
        word2idx=word2idx,
        idx2word=idx2word,
        feature_extractor=feature_extractor,
        beam_width=10,
        max_len=100
    ))




📸 Image #1
Generated: the lungs are clear bilaterally specifically no evidence of focal consolidation pneumothorax or pleural effusion cardio mediastinal silhouette is unremarkable visualized osseous structures of the thorax are without acute abnormality no acute cardiopulmonary abnormality

📸 Image #2
Generated: the lungs are clear bilaterally specifically no evidence of focal consolidation pneumothorax or pleural effusion cardio mediastinal silhouette is unremarkable visualized osseous structures of the thorax are without acute abnormality no acute cardiopulmonary abnormality

📸 Image #3
Generated: no information normal heart size normal pulmonary vasculature normal mediastinal contours lung degree is clear no airspace disease no pulmonary edema no of pleural effusions no of active cardiopulmonary disease

📸 Image #4
Generated: the cardiomediastinal silhouette is within normal limits for size and contour the lungs are normally inflated without evidence of focal airspace disease pleu

In [24]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

In [25]:
def compute_bleu_scores(reference, candidate):
    reference = [reference.lower().split()]
    candidate = candidate.lower().split()
    smoothie = SmoothingFunction().method1

    bleu1 = sentence_bleu(reference, candidate, weights=(1, 0, 0, 0), smoothing_function=smoothie)
    bleu2 = sentence_bleu(reference, candidate, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothie)
    bleu3 = sentence_bleu(reference, candidate, weights=(0.33, 0.33, 0.33, 0), smoothing_function=smoothie)
    bleu4 = sentence_bleu(reference, candidate, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie)

    return {
        "BLEU-1": bleu1,
        "BLEU-2": bleu2,
        "BLEU-3": bleu3,
        "BLEU-4": bleu4,
        "Average": (bleu1 + bleu2 + bleu3 + bleu4) / 4
    }


In [26]:
print("📊 Evaluating BLEU score on validation set...\n")

total_scores = {
    "BLEU-1": 0,
    "BLEU-2": 0,
    "BLEU-3": 0,
    "BLEU-4": 0,
    "Average": 0
}

num_samples = 20  # or any number <= len(valid_dataset_reports)
for i in range(num_samples):
    image, report_tensor = valid_dataset_reports[i]
    ground_truth = decode_report(report_tensor.tolist())  # make sure `decode_report` is defined
    generated = beam_search(image, encoder, lstm, decoder, word2idx, idx2word, feature_extractor)

    scores = compute_bleu_scores(ground_truth, generated)
    for key in total_scores:
        total_scores[key] += scores[key]

    print(f"📸 Image #{i+1}")
    print(f"📝 Ground Truth : {ground_truth}")
    print(f"🤖 Generated    : {generated}")
    print(f"🔵 BLEU Score   : {scores['Average']:.4f}")
    print("—" * 80)

# 🔚 Final Average
for key in total_scores:
    total_scores[key] /= num_samples

print("✅ Average BLEU scores over", num_samples, "samples:")
for key, val in total_scores.items():
    print(f"{key}: {val:.4f}")


📊 Evaluating BLEU score on validation set...

📸 Image #1
📝 Ground Truth : <start> the cardiac silhouette and mediastinum size are within normal limits there is no pulmonary edema there is no focal consolidation there are no of pleural effusion there is no evidence of pneumothorax normal chest <end>
🤖 Generated    : the lungs are clear bilaterally specifically no evidence of focal consolidation pneumothorax or pleural effusion cardio mediastinal silhouette is unremarkable visualized osseous structures of the thorax are without acute abnormality no acute cardiopulmonary abnormality
🔵 BLEU Score   : 0.1984
————————————————————————————————————————————————————————————————————————————————
📸 Image #2
📝 Ground Truth : <start> the cardiac silhouette and mediastinum size are within normal limits there is no pulmonary edema there is no focal consolidation there are no of pleural effusion there is no evidence of pneumothorax normal chest <end>
🤖 Generated    : the lungs are clear bilaterally speci